# Week 45, Day 2: Binary trees

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w045d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def size(node):
    if node is None:
        return 0
    return 1 + size(node.left) + size(node.right)

def height(node):
    if node is None:
        return -1
    return 1 + max(height(node.left), height(node.right))

t = TreeNode(1, TreeNode(2, TreeNode(4), TreeNode(5)), TreeNode(3, None, TreeNode(6, TreeNode(7))))
print(size(t), height(t), height(None), height(TreeNode(9)))

In [ ]:
tree = ["A", "B", "C", "D", "E", "F"]

def children(i):
    return [tree[j] for j in (2 * i + 1, 2 * i + 2) if j < len(tree)]

def parent(i):
    return tree[(i - 1) // 2] if i > 0 else None

for i, name in enumerate(tree):
    print(name, "children:", children(i), "parent:", parent(i))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mirror(node):
    """Swap left and right everywhere, in place. Returns the node."""
    if node is not None:
        node.left, node.right = mirror(node.right), mirror(node.left)
    return node

def same(a, b):
    if a is None or b is None:
        return a is b                         # both empty, or a mismatch
    return a.val == b.val and same(a.left, b.left) and same(a.right, b.right)

def is_symmetric(root):
    def mirrored(a, b):
        if a is None or b is None:
            return a is b
        return a.val == b.val and mirrored(a.left, b.right) and mirrored(a.right, b.left)
    return mirrored(root, root)

t = TreeNode(1, TreeNode(2, TreeNode(3), TreeNode(4)), TreeNode(2, TreeNode(4), TreeNode(3)))
print(is_symmetric(t), same(t.left, t.right), same(mirror(TreeNode(2, TreeNode(3), TreeNode(4))), t.right))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def checked_height(node):
    """The height, or -2 if any subtree is unbalanced."""
    if node is None:
        return -1
    left = checked_height(node.left)
    right = checked_height(node.right)
    if left == -2 or right == -2 or abs(left - right) > 1:
        return -2
    return 1 + max(left, right)

def is_balanced(root):
    return checked_height(root) != -2

ok = TreeNode(1, TreeNode(2, TreeNode(4)), TreeNode(3))
chain = TreeNode(1, TreeNode(2, TreeNode(3)))
print(is_balanced(ok), is_balanced(chain), is_balanced(None))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def evaluate(node):
    if node.left is None and node.right is None:
        return node.val
    a, b = evaluate(node.left), evaluate(node.right)
    return a + b if node.val == "+" else a - b if node.val == "-" else a * b if node.val == "*" else a / b

def show(node):
    if node.left is None and node.right is None:
        return str(node.val)
    return f"({show(node.left)} {node.val} {show(node.right)})"

e = TreeNode("*", TreeNode("+", TreeNode(2), TreeNode(3)), TreeNode(4))
print(show(e), "=", evaluate(e))

> **Key idea.** A binary tree's nodes have at most two children, `left` and `right`, and the empty tree is `None`, so most functions start with `if node is None` and combine answers from both sides. Shapes: full (0 or 2 children), perfect (every level full), complete (filled level by level from the left), balanced (subtree heights differ by at most 1 everywhere), degenerate (a chain). Height is at least about log₂ n and at most n − 1, and root-to-leaf work costs O(height). A complete tree fits in a list: children at `2i + 1` and `2i + 2`, parent at `(i − 1) // 2`. Check balance in O(n) by returning heights with a sentinel for "unbalanced".

### Exercise: From a list, and back

Write `from_levels(values)`: build a binary tree from a level-order list with `None` for missing nodes (the children of a missing node aren't listed): `[1, 2, 3, None, 4]` is 1 with children 2 and 3, and 2 has only a right child, 4. Return the root (`None` for an empty list or `[None]`). Then `to_levels(root)`, the reverse, with trailing `None`s removed. Use a `deque` (Week 43).

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def from_levels(values):
    ___

def to_levels(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_levels():\n    \"converting\"\n    t = from_levels([1, 2, 3, None, 4])\n    assert t.val == 1 and t.left.val == 2 and t.right.val == 3 and t.left.left is None and t.left.right.val == 4 and t.right.left is None\n    for vals in [[1, 2, 3, None, 4], [1, None, 2, None, 3], [5], [], [1, 2, 3, 4, 5, 6, 7], [3, 9, 20, None, None, 15, 7]]:\n        assert to_levels(from_levels(vals)) == vals, vals\n    assert from_levels([None]) is None and to_levels(None) == []")

### Exercise: What shape is it?

Write `is_full(root)` (every node has 0 or 2 children; the empty tree is full), `is_perfect(root)` (full and every leaf at the same depth: check that its size is `2 ** (height + 1) - 1`), and `count_leaves(root)`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def is_full(root):
    ___

def is_perfect(root):
    ___

def count_leaves(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_shape():\n    \"shapes\"\n    perfect = T(1, T(2, T(4), T(5)), T(3, T(6), T(7)))\n    full = T(1, T(2, T(4), T(5)), T(3))\n    neither = T(1, T(2, T(4)), T(3))\n    assert is_full(perfect) and is_perfect(perfect) and count_leaves(perfect) == 4\n    assert is_full(full) and not is_perfect(full) and count_leaves(full) == 3\n    assert not is_full(neither) and not is_perfect(neither) and count_leaves(neither) == 2\n    assert is_full(None) and is_perfect(None) and count_leaves(None) == 0 and is_perfect(T(1))")

### Exercise: A complete tree in a list

A complete binary tree is stored in a list, level by level. Write `children(tree, i)` (the **indexes** of the children of position `i` that exist), `parent(i)` (the parent's index, or `None` for the root), `depth(i)` (the depth of position `i`) and `path_up(i)` (the indexes from `i` up to the root).

In [ ]:
def children(tree, i):
    ___

def parent(i):
    ___

def depth(i):
    ___

def path_up(i):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_array():\n    \"positions\"\n    tree = list(\"ABCDEFGHIJ\")\n    assert children(tree, 0) == [1, 2] and children(tree, 4) == [9] and children(tree, 5) == [] and children(tree, 3) == [7, 8]\n    assert parent(0) is None and parent(9) == 4 and parent(2) == 0\n    assert [depth(i) for i in [0, 1, 2, 3, 6, 7, 14]] == [0, 1, 1, 2, 2, 3, 3]\n    assert path_up(9) == [9, 4, 1, 0] and path_up(0) == [0]")

### Exercise: Mirrors

Write `mirror(root)` (swap left and right at every node, in place, returning the root), `same(a, b)` (the same shape and values) and `is_symmetric(root)` (a mirror image of itself around its centre).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mirror(root):
    ___

def same(a, b):
    ___

def is_symmetric(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("T = TreeNode\ndef test_mirror():\n    \"mirrors\"\n    t = T(1, T(2, T(3), T(4)), T(5))\n    m = mirror(T(1, T(2, T(3), T(4)), T(5)))\n    assert same(m, T(1, T(5), T(2, T(4), T(3)))) and not same(m, t) and same(None, None) and not same(T(1), None)\n    assert same(mirror(mirror(T(1, T(2, T(3), T(4)), T(5)))), t)\n    assert is_symmetric(T(1, T(2, T(3), T(4)), T(2, T(4), T(3)))) and not is_symmetric(T(1, T(2, None, T(3)), T(2, None, T(3)))) and is_symmetric(None)")

### Exercise: Balanced, in one pass

Write `is_balanced(root)` in O(n): at every node, the heights of the left and right subtrees differ by at most 1. Use one recursive helper that returns the height, or `-2` if anything below is unbalanced, and never computes a height twice. The checks count how many times your code looks at each node's `left`: on a chain of 500 nodes, a check that recomputes heights looks hundreds of times more often.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def is_balanced(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sys\nT = TreeNode\nclass Counting(TreeNode):\n    reads = 0\n    def __getattribute__(self, name):\n        if name == \"left\":\n            Counting.reads += 1\n        return object.__getattribute__(self, name)\ndef test_balanced():\n    \"balance\"\n    assert is_balanced(T(1, T(2, T(4)), T(3))) and not is_balanced(T(1, T(2, T(3)))) and is_balanced(None)\n    assert not is_balanced(T(1, T(2, T(4, T(8)), T(5)), T(3)))\n    assert is_balanced(T(1, T(2, T(4), T(5)), T(3, T(6))))\ndef test_linear():\n    \"each node visited once\"\n    nodes = None\n    for i in range(500):\n        nodes = Counting(i, nodes, None)\n    Counting.reads = 0\n    is_balanced(nodes)\n    assert Counting.reads <= 1000, f\"{Counting.reads} reads of .left for 500 nodes\"\n    perfect = None\n    def build(d):\n        return None if d < 0 else Counting(d, build(d - 1), build(d - 1))\n    p = build(9)\n    Counting.reads = 0\n    assert is_balanced(p) and Counting.reads <= 2 * 1023")

### Exercise: Expression trees

An expression tree's leaves hold numbers and its internal nodes hold `"+"`, `"-"`, `"*"` or `"/"`. Write `evaluate(node)`, `infix(node)` (fully bracketed, like `((2 + 3) * 4)`, but no brackets around a single number), and `from_postfix(tokens)`, which builds the tree from postfix tokens like `["2", "3", "+", "4", "*"]` with a stack of subtrees (numbers become ints).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def evaluate(node):
    ___

def infix(node):
    ___

def from_postfix(tokens):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_expr():\n    \"expressions\"\n    t = from_postfix(\"2 3 + 4 *\".split())\n    assert t.val == \"*\" and t.right.val == 4 and evaluate(t) == 20 and infix(t) == \"((2 + 3) * 4)\"\n    u = from_postfix(\"10 2 3 * -\".split())\n    assert evaluate(u) == 4 and infix(u) == \"(10 - (2 * 3))\"\n    assert evaluate(from_postfix([\"7\"])) == 7 and infix(from_postfix([\"7\"])) == \"7\"\n    assert evaluate(from_postfix(\"8 2 / 3 /\".split())) == 8 / 2 / 3")

### Bug hunt: A swap that loses a side

This `mirror` sets `left` first, then uses the **new** `left` to set `right`, so the original left subtree is lost and both sides end up the same: this prints `[1, 3, 3]`. Swap both at once, so it prints `[1, 3, 2]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mirror(node):
    if node is not None:
        node.left = mirror(node.right)
        node.right = mirror(node.left)
    return node

t = mirror(TreeNode(1, TreeNode(2), TreeNode(3)))
print([t.val, t.left.val, t.right.val])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"swap together\"\n    expect_output(\"[1, 3, 2]\")")

### Bug hunt: Shape doesn't matter?

`same` compares only the values it meets, and treats a missing child as matching anything, so two trees with the same values in different places are called the same: this prints `True`. Compare the shape too, so it prints `False`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def same(a, b):
    if a is None or b is None:
        return True
    return a.val == b.val and same(a.left, b.left) and same(a.right, b.right)

print(same(TreeNode(1, TreeNode(2)), TreeNode(1, None, TreeNode(2))))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shape matters\"\n    expect_output(\"False\")\n    assert same(TreeNode(1, TreeNode(2)), TreeNode(1, TreeNode(2))) and same(None, None)")

### Bug hunt: The empty tree has height 0?

This height function says an empty tree has height 0, so a single node gets height 1, and the balance check is fooled: a node with one child that has a child looks balanced. This prints `True`. Give the empty tree height −1, so it prints `False`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def height(node):
    if node is None:
        return 0
    return 1 + max(height(node.left), height(node.right))

def balanced_here(node):
    return abs(height(node.left) - height(node.right)) <= 1

t = TreeNode(1, TreeNode(2, TreeNode(3)))
print(balanced_here(t) and balanced_here(t.left) and height(TreeNode(9)) == 1)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"-1 for empty\"\n    expect_output(\"False\")\n    assert height(None) == -1 and height(TreeNode(9)) == 0")

### Puzzle: Is it symmetric?

Arrange the lines into the helper that checks whether two subtrees are mirror images. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Answer without running code.

p = ___   # the most nodes a binary tree of height 4 can have
q = ___   # the smallest possible height for a binary tree of 100 nodes
r = ___   # the largest possible height for a binary tree of 100 nodes
s = ___   # in a list-stored complete tree, the index of the parent of position 11
t = ___   # in a list-stored complete tree, the indexes of the children of position 5

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 31 and q == 6 and r == 99, \"p, q and r\"\n    assert s == 5 and t == [11, 12], \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the longest path, and the best path

Two classic problems, each solved in **one** O(n) pass by returning more than one thing from a recursive helper:

1. `diameter(root)`: the number of edges on the longest path between **any** two nodes (the path needn't pass through the root). Hint: at each node, the longest path through it goes down its left side and down its right side; the helper returns the height, and updates the best diameter seen as it goes.
2. `best_path_sum(root)`: the largest sum of values along any path between two nodes (a path may be a single node; values may be negative; the tree has at least one node). Hint: the helper returns the best sum of a path going **down** from this node (choosing at most one side, and ignoring a side whose best is negative), while the answer is updated with "this node plus the best of each side". (To let the helper update the best answer, keep it in a one-item list, `best = [0]`, and change `best[0]`; Phase 5 shows the `nonlocal` keyword, which does this more directly.)

The checks use trees with up to 600 nodes and compare with a slow version.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def diameter(root):
    ___

def best_path_sum(root):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nT = TreeNode\ndef nodes(n):\n    return [] if n is None else [n] + nodes(n.left) + nodes(n.right)\ndef neighbours(root):\n    adj = {}\n    for n in nodes(root):\n        adj.setdefault(id(n), [])\n        for c in (n.left, n.right):\n            if c is not None:\n                adj[id(n)].append(c)\n                adj.setdefault(id(c), []).append(n)\n    return adj\ndef slow(root):\n    adj = neighbours(root)\n    best_len, best_sum = 0, max(n.val for n in nodes(root))\n    for start in nodes(root):\n        stack = [(start, None, 0, start.val)]\n        while stack:\n            node, came, length, total = stack.pop()\n            best_len, best_sum = max(best_len, length), max(best_sum, total)\n            for nxt in adj[id(node)]:\n                if nxt is not came:\n                    stack.append((nxt, node, length + 1, total + nxt.val))\n    return best_len, best_sum\ndef random_tree(rng, n):\n    root = T(rng.randint(-10, 10))\n    all_nodes = [root]\n    for _ in range(n - 1):\n        parent = rng.choice(all_nodes)\n        child = T(rng.randint(-10, 10))\n        if parent.left is None and (parent.right is not None or rng.random() < 0.5):\n            parent.left = child\n        elif parent.right is None:\n            parent.right = child\n        else:\n            continue\n        all_nodes.append(child)\n    return root\ndef test_small():\n    \"small trees\"\n    t = T(1, T(2, T(4), T(5)), T(3))\n    assert diameter(t) == 3 and diameter(T(1)) == 0 and diameter(None) == 0\n    assert best_path_sum(T(-10, T(9), T(20, T(15), T(7)))) == 42 and best_path_sum(T(-3)) == -3 and best_path_sum(T(2, T(-1), T(-2))) == 2\ndef test_random():\n    \"against a slow version\"\n    rng = random.Random(4)\n    for _ in range(40):\n        t = random_tree(rng, rng.randint(1, 25))\n        assert (diameter(t), best_path_sum(t)) == slow(t)\n    big = random_tree(rng, 600)\n    assert diameter(big) >= 1")

## Warm-up

With binary trees, the **empty tree** is `None`, and that makes the recursion beautifully uniform: every function's base case is `node is None`, and every node simply asks its two children.

## Left and right

```text
            1
          /   \
         2     3            each node: val, left, right
        / \     \           a missing child is None
       4   5     6
                /
               7
```

Unlike yesterday's general trees, the two children have **positions**: a node with only a right child is different from one with only a left child. That's why `TreeNode` has two named attributes rather than a list. (The name `TreeNode` with `val`, `left` and `right` is the convention most books and practice sites use.)

Binary trees are everywhere once you look: an expression like `(2 + 3) * 4` is a binary tree with operators inside and numbers at the leaves; a "twenty questions" game is a tree of yes/no questions; tournaments are binary trees of matches; and Huffman codes, used to compress files, are built from a binary tree.

## Shapes and sizes

| Shape | Meaning |
|---|---|
| **full** | every node has 0 or 2 children |
| **perfect** | full, and all leaves at the same depth: every level completely filled |
| **complete** | every level filled except perhaps the last, which is filled from the left |
| **balanced** (height-balanced) | at every node, the heights of the two subtrees differ by at most 1 |
| **degenerate** | every node has at most one child: really a linked list |

How height and size relate is the key to everything that follows:

- Level `d` holds at most `2^d` nodes (1, 2, 4, 8, ...).
- A perfect tree of height `h` has `2^(h+1) − 1` nodes: about half of them are leaves.
- So the **smallest** possible height for `n` nodes is about `log₂ n` (exactly `⌊log₂ n⌋`), and the largest is `n − 1` (degenerate).

A balanced tree of a million nodes is only about 20 levels tall; a degenerate one is a million. Every operation that walks from the root to a leaf costs O(height), so keeping trees balanced (next week) is the difference between O(log n) and O(n).

## A tree in a list

A **complete** binary tree can be stored without any node objects: put the nodes in a list, level by level, left to right. Then positions do the work of links:

```text
             0: A
           /      \
       1: B        2: C            children of i:  2i + 1 and 2i + 2
       /  \        /                parent of i:    (i - 1) // 2
    3: D  4: E   5: F

    list: [A, B, C, D, E, F]
```

No links, no wasted space, and finding a parent or child is one calculation. The catch: it only works for complete trees, since a gap would waste a slot (and a degenerate tree would need 2^n slots). Heaps, a few weeks from now, are exactly complete binary trees stored like this.

For trees with gaps, a common written form (used by many practice sites) is the level order **with `None` for missing children**: `[1, 2, 3, 4, 5, None, 6]` means the tree above without the 7. You'll write the converters in the practice.

## Recursion with None

Most binary-tree functions follow one template: handle `None`, then combine the answers from `node.left` and `node.right`.

The tuple assignment in `mirror` matters: both right-hand sides are worked out **before** either attribute changes, so neither subtree is lost.

**Balance in O(n).** The obvious balance check (at every node, compute both subtrees' heights) recomputes heights over and over: O(n²) on a degenerate tree. Better: one function that returns the height **and** spots imbalance on the way up, using `-2` (an impossible height) as a signal that something below is unbalanced:

Each node is visited once: O(n). This "return more than one thing on the way up" idea solves many harder problems, including today's challenge.

## Expression trees

An arithmetic expression is a binary tree: internal nodes are operators, leaves are numbers. Evaluating it is a recursion that works out both children, then applies the operator (Week 43's postfix evaluation was exactly this, done with a stack).

## Practice

## Quiz

## Daily challenge

## Before you go

Draw a binary tree of 7 nodes that is complete but not perfect, one that is full but not complete, and one that is balanced but not complete. Tomorrow: depth-first traversals, the three orders (pre, in and post) in which a binary tree can be walked, recursively and with an explicit stack.